In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import Transformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

In [2]:
%store -r cn0dbhz_merge_df
%store -r satcount_merge_df
%store -r elevationdeg_merge_df
%store -r accmag_merge_df
%store -r gyromag_merge_df

In [3]:
print('Cn0DbHz Shape: ')
print(cn0dbhz_merge_df.shape)
print()
print('SatCount Shape: ')
print(satcount_merge_df.shape)
print()
print('SvElevationDegrees Shape: ')
print(elevationdeg_merge_df.shape)
print()
print('AccMag Shape: ')
print(accmag_merge_df.shape)
print()
print('GyroMag Shape: ')
print(gyromag_merge_df.shape)
print()

Cn0DbHz Shape: 
(153631, 7)

SatCount Shape: 
(153631, 7)

SvElevationDegrees Shape: 
(153631, 7)

AccMag Shape: 
(153631, 7)

GyroMag Shape: 
(153631, 7)



In [4]:
merge_df = cn0dbhz_merge_df.merge(satcount_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'SatCount']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(elevationdeg_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'SvElevationDegrees']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(accmag_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'AccMag']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(gyromag_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'GyroMag']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')

In [5]:
print(merge_df)

                   drive_id        device  UnixTimeMillis    Cn0DbHz  \
0       2020-06-10-US-MTV-2  GooglePixel4   1591819502443  30.987097   
1       2020-06-10-US-MTV-2  GooglePixel4   1591819503443  31.315625   
2       2020-06-10-US-MTV-2  GooglePixel4   1591819509443  32.503226   
3       2020-06-10-US-MTV-2  GooglePixel4   1591819514443  31.681250   
4       2020-06-10-US-MTV-2  GooglePixel4   1591819515443  31.216667   
...                     ...           ...             ...        ...   
153626  2021-12-28-US-MTV-1     XiaomiMi8   1640724239000  25.916074   
153627  2021-12-28-US-MTV-1     XiaomiMi8   1640724240000  25.838720   
153628  2021-12-28-US-MTV-1     XiaomiMi8   1640724242000  25.539480   
153629  2021-12-28-US-MTV-1     XiaomiMi8   1640724243000  25.792816   
153630  2021-12-28-US-MTV-1     XiaomiMi8   1640724244000  25.816935   

        ErrorXEcefMeters  ErrorYEcefMeters  ErrorZEcefMeters  SatCount  \
0               0.528437          2.542246          2.295396 

In [6]:
print(merge_df.isna().sum())

drive_id              0
device                0
UnixTimeMillis        0
Cn0DbHz               0
ErrorXEcefMeters      0
ErrorYEcefMeters      0
ErrorZEcefMeters      0
SatCount              0
SvElevationDegrees    1
AccMag                0
GyroMag               0
dtype: int64


In [7]:
# drop null values

merge_df = merge_df.dropna(subset=['SvElevationDegrees'])

print(merge_df.isna().sum())
print()
print('Merge Shape: ')
print(merge_df.shape)
print()

drive_id              0
device                0
UnixTimeMillis        0
Cn0DbHz               0
ErrorXEcefMeters      0
ErrorYEcefMeters      0
ErrorZEcefMeters      0
SatCount              0
SvElevationDegrees    0
AccMag                0
GyroMag               0
dtype: int64

Merge Shape: 
(153630, 11)



In [8]:
X = merge_df[['Cn0DbHz', 'SatCount', 'SvElevationDegrees', 'AccMag', 'GyroMag']]
y = pd.Series(merge_df['ErrorXEcefMeters'])

In [9]:
# split train and test data with 80% and 20% respectively

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# train multiple linear regression model

modelx = LinearRegression()

modelx.fit(X_train, y_train)

LinearRegression()

In [11]:
print("Coefficients:", modelx.coef_)  
print("Intercept:", modelx.intercept_)

Coefficients: [-0.09333673  0.03536005  0.04953196  0.00790801 -0.62513148]
Intercept: -0.8182061389696682


In [12]:
y_train_pred = modelx.predict(X_train)

In [13]:
y_test_pred = modelx.predict(X_test)

In [14]:
# metrics for train data

print('Goodness of Fit of Model \tTrain Dataset')
print("Explained Variance (R^2) \t:", modelx.score(X_train, y_train)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

def mean_sq_err(actual, predicted):
    '''Returns the Mean Squared Error of actual and predicted values'''
    return np.mean(np.square(np.array(actual) - np.array(predicted))) 

mse = mean_sq_err(y_train, y_train_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Train Dataset
Explained Variance (R^2) 	: 0.02746374331457513
Mean Squared Error (MSE) 	: 6.842115070540917
Root Mean Squared Error (RMSE) 	: 2.6157436935871443


In [15]:
# metrics for test data

print('Goodness of Fit of Model \tTest Dataset')
print("Explained Variance (R^2) \t:", modelx.score(X_test, y_test)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

def mean_sq_err(actual, predicted):
    '''Returns the Mean Squared Error of actual and predicted values'''
    return np.mean(np.square(np.array(actual) - np.array(predicted))) 

mse = mean_sq_err(y_test, y_test_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Test Dataset
Explained Variance (R^2) 	: 0.02860996206304156
Mean Squared Error (MSE) 	: 6.851335253426273
Root Mean Squared Error (RMSE) 	: 2.617505540285688


In [16]:
X = merge_df[['Cn0DbHz', 'SatCount', 'SvElevationDegrees', 'AccMag', 'GyroMag']]
y = pd.Series(merge_df['ErrorYEcefMeters'])

In [17]:
# split train and test data with 80% and 20% respectively

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [18]:
# train multiple linear regression model

modely = LinearRegression()

modely.fit(X_train, y_train)

LinearRegression()

In [19]:
print("Coefficients:", modely.coef_)  
print("Intercept:", modely.intercept_)

Coefficients: [-0.10737351 -0.04140318 -0.09486983  0.0030852  -0.29106138]
Intercept: 7.239996294845434


In [20]:
y_train_pred = modely.predict(X_train)

In [21]:
y_test_pred = modely.predict(X_test)

In [22]:
# metrics for train data

print('Goodness of Fit of Model \tTrain Dataset')
print("Explained Variance (R^2) \t:", modely.score(X_train, y_train)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

def mean_sq_err(actual, predicted):
    '''Returns the Mean Squared Error of actual and predicted values'''
    return np.mean(np.square(np.array(actual) - np.array(predicted))) 

mse = mean_sq_err(y_train, y_train_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Train Dataset
Explained Variance (R^2) 	: 0.015468218196321715
Mean Squared Error (MSE) 	: 10.594415250980227
Root Mean Squared Error (RMSE) 	: 3.254906335208469


In [23]:
# metrics for test data

print('Goodness of Fit of Model \tTest Dataset')
print("Explained Variance (R^2) \t:", modely.score(X_test, y_test)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

def mean_sq_err(actual, predicted):
    '''Returns the Mean Squared Error of actual and predicted values'''
    return np.mean(np.square(np.array(actual) - np.array(predicted))) 

mse = mean_sq_err(y_test, y_test_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Test Dataset
Explained Variance (R^2) 	: 0.016505681902843405
Mean Squared Error (MSE) 	: 10.508518707400055
Root Mean Squared Error (RMSE) 	: 3.2416845477930227


In [24]:
X = merge_df[['Cn0DbHz', 'SatCount', 'SvElevationDegrees', 'AccMag', 'GyroMag']]
y = pd.Series(merge_df['ErrorZEcefMeters'])

In [25]:
# split train and test data with 80% and 20% respectively

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [26]:
# train multiple linear regression model

modelz = LinearRegression()

modelz.fit(X_train, y_train)

LinearRegression()

In [27]:
print("Coefficients:", modelz.coef_)  
print("Intercept:", modelz.intercept_)

Coefficients: [ 0.02729187 -0.04032544  0.03465186 -0.04162998 -0.08146855]
Intercept: 0.12996202965740233


In [28]:
y_train_pred = modelz.predict(X_train)

In [29]:
y_test_pred = modelz.predict(X_test)

In [30]:
# metrics for train data

print('Goodness of Fit of Model \tTrain Dataset')
print("Explained Variance (R^2) \t:", modelz.score(X_train, y_train)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

def mean_sq_err(actual, predicted):
    '''Returns the Mean Squared Error of actual and predicted values'''
    return np.mean(np.square(np.array(actual) - np.array(predicted))) 

mse = mean_sq_err(y_train, y_train_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Train Dataset
Explained Variance (R^2) 	: 0.011654963175135036
Mean Squared Error (MSE) 	: 10.286595598633307
Root Mean Squared Error (RMSE) 	: 3.207272298797423


In [31]:
# metrics for test data

print('Goodness of Fit of Model \tTest Dataset')
print("Explained Variance (R^2) \t:", modelz.score(X_test, y_test)) # tells you how accurate your model is fitting the data, the closer it is to 1, the better the fit.

def mean_sq_err(actual, predicted):
    '''Returns the Mean Squared Error of actual and predicted values'''
    return np.mean(np.square(np.array(actual) - np.array(predicted))) 

mse = mean_sq_err(y_test, y_test_pred)
print("Mean Squared Error (MSE) \t:", mse)
print("Root Mean Squared Error (RMSE) \t:", np.sqrt(mse)) 

Goodness of Fit of Model 	Test Dataset
Explained Variance (R^2) 	: 0.009160246324483534
Mean Squared Error (MSE) 	: 10.399709248436407
Root Mean Squared Error (RMSE) 	: 3.2248580198880705


In [32]:
%store -r merge_test_df

In [34]:
features_test_df = merge_test_df.drop(columns=['UnixTimeMillis', 'tripId', 'WlsPositionXEcefMeters','WlsPositionYEcefMeters', 'WlsPositionZEcefMeters'])

predicted_errorx = modelx.predict(features_test_df)
predicted_errory = modely.predict(features_test_df)
predicted_errorz = modelz.predict(features_test_df)

print('predicted_errorx: ')
print(predicted_errorx)
print()
print('predicted_errory: ')
print(predicted_errory)
print()
print('predicted_errorz: ')
print(predicted_errorz)
print()

predicted_errorx: 
[-0.05992813 -0.08112351 -0.09225441 ... -1.31316348 -1.23378287
 -1.29589228]

predicted_errory: 
[-0.91707884 -0.93923388 -0.83661018 ... -0.52052889 -0.66323705
 -0.49992187]

predicted_errorz: 
[0.22096886 0.22176145 0.1906026  ... 1.00490818 1.11589463 1.00045896]



In [41]:
submission_ecef = pd.DataFrame()
submission_ecef['MeasurementX_Corr'] = merge_test_df['WlsPositionXEcefMeters'] + predicted_errorx
submission_ecef['MeasurementY_Corr'] = merge_test_df['WlsPositionYEcefMeters'] + predicted_errory
submission_ecef['MeasurementZ_Corr'] = merge_test_df['WlsPositionZEcefMeters'] + predicted_errorz

In [42]:
transformer = Transformer.from_crs("EPSG:4978", "EPSG:4326", always_xy=True)

lon, lat, alt = transformer.transform(
    submission_ecef['MeasurementX_Corr'].values, 
    submission_ecef['MeasurementY_Corr'].values,
    submission_ecef['MeasurementZ_Corr'].values,
)

In [43]:
submission = pd.DataFrame()
submission['tripId'] = merge_test_df['tripId']
submission['UnixTimeMillis'] = merge_test_df['UnixTimeMillis']
submission['LatitudeDegrees'] = lat
submission['LongitudeDegrees'] = lon

print(submission)
print()
submission.to_csv('./SUBMISSIONS/submission.csv', index=False)

                                          tripId  UnixTimeMillis  \
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650832999   
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650833999   
2      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650834999   
3      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650835999   
4      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650836999   
...                                          ...             ...   
66092           2022-04-25-US-OAK-2/GooglePixel4   1650927742650   
66093           2022-04-25-US-OAK-2/GooglePixel4   1650927743642   
66094           2022-04-25-US-OAK-2/GooglePixel4   1650927744651   
66095           2022-04-25-US-OAK-2/GooglePixel4   1650927745640   
66096           2022-04-25-US-OAK-2/GooglePixel4   1650927746632   

       LatitudeDegrees  LongitudeDegrees  
0            37.395825       -122.102963  
1            37.395848       -122.102982  
2            37.395832       -122.102997  
3          

In [44]:
print(merge_test_df.isna().sum())

UnixTimeMillis            0
Cn0DbHz                   0
SatCount                  0
SvElevationDegrees        0
tripId                    0
AccMag                    0
GyroMag                   0
WlsPositionXEcefMeters    0
WlsPositionYEcefMeters    0
WlsPositionZEcefMeters    0
dtype: int64
